# 🖥️ Notebook 10 — Streamlit App & Demo

## 🎯 Objective
Provide an interactive **Streamlit** app for heart-disease risk scoring. The app loads the **best model
pipeline** (preprocessing + estimator), allows users to input clinical features, and returns a
**predicted probability** and **decision label** at a selectable threshold.

## 🔗 Inputs
- Best model pointer: `reports/best_model.json`
- Model artifact: `models/heart_model_<run_id>.pkl`
- Raw feature schema (same as training)

## 📦 Outputs
- `app/streamlit_app.py` — interactive Streamlit UI
- (Optional) CSV with demo batch scoring

## 🧭 What you’ll do
1. Verify environment & dependencies
2. Load the **best model** artifact
3. Define the **canonical input schema** for inference
4. Create a **Streamlit UI** with typed widgets + threshold slider
5. Run locally (localhost) or in **Colab** via `pyngrok`
6. (Optional) Batch-score a CSV for demo purposes

> Tip: This app is for **demo & exploration**. For production APIs, see Notebook 09 (FastAPI).




## 0) Setup & imports

In [1]:
# (Colab only) mount & cd
IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction

Mounted at /content/drive
/content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction


In [2]:
# Install runtime deps if needed
!pip -q install streamlit pyngrok

import os, sys, json, time, warnings, threading, subprocess, textwrap
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.append(os.getcwd())

import numpy as np
import pandas as pd
import joblib

# Project paths
MODELS_DIR  = Path("models");  MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)
APP_DIR     = Path("app");     APP_DIR.mkdir(parents=True, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 90.8 MB/s eta 0:00:00


## 1) Resolve the best model and define the inference scheme

In [5]:
# Discover best run_id from pointer (fallback: newest)
best_meta_path = REPORTS_DIR / "best_model.json"
if best_meta_path.exists():
    best_meta = json.loads(best_meta_path.read_text())
    RUN_ID = best_meta["best_run_id"]
else:
    candidates = sorted(MODELS_DIR.glob("heart_model_*.pkl"), key=lambda p: p.stat().st_mtime, reverse=True)
    assert candidates, "No model artifacts found in models/."
    RUN_ID = candidates[0].stem.replace("heart_model_", "")

artifact_path = MODELS_DIR / f"heart_model_{RUN_ID}.pkl"
print("Using model:", artifact_path.name)

# Canonical raw features expected by the pipeline (pre-derivation)
FEATURES_ORDER = [
    "age","sex","cp","trestbps","chol","fbs","restecg",
    "thalach","exang","oldpeak","slope","ca","thal"
]
DTYPES = {
    "age": float, "sex": int, "cp": int, "trestbps": float, "chol": float,
    "fbs": int, "restecg": int, "thalach": float, "exang": int,
    "oldpeak": float, "slope": int, "ca": float, "thal": int
}


Using model: heart_model_logreg_tuned_seed42_1761367619.pkl


## 2) Create the Streamlit app file (app/streamlit_app.py)

In [7]:
streamlit_code = f"""
import streamlit as st
import pandas as pd
import numpy as np
import json
from pathlib import Path
import joblib

# --- Paths & pointers ---
MODELS_DIR = Path("models")
REPORTS_DIR = Path("reports")
BEST_PATH = REPORTS_DIR / "best_model.json"

# --- Schema (must match training) ---
FEATURES_ORDER = {FEATURES_ORDER}
DTYPES_DICT = {DTYPES} # Define DTYPES as a dictionary first

def load_best_model():
    if BEST_PATH.exists():
        best = json.loads(BEST_PATH.read_text())
        run = best["best_run_id"]
    else:
        pkls = sorted(MODELS_DIR.glob("heart_model_*.pkl"), key=lambda p: p.stat().st_mtime, reverse=True)
        if not pkls:
            st.error("No model artifacts found in models/.")
            st.stop()
        run = pkls[0].stem.replace("heart_model_", "")
    model = joblib.load(MODELS_DIR / f"heart_model_{{run}}.pkl")
    return run, model

def validate_and_cast(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in FEATURES_ORDER if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {{missing}}")
    df = df[FEATURES_ORDER].copy()
    for c, t in DTYPES_DICT.items(): # Use the defined dictionary
        # Cast using numpy/pandas dtype strings
        if 'int' in str(t):
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64').astype('float').astype(int)
        elif 'float' in str(t):
            df[c] = pd.to_numeric(df[c], errors='coerce').astype(float)
        else:
            df[c] = df[c].astype(t)
    return df

st.set_page_config(page_title="Heart Disease Risk — Streamlit Demo", layout="centered")
st.title("❤️ Heart Disease Risk — Streamlit Demo")

RUN_ID, PIPE = load_best_model()
st.caption(f"Using model run: {{RUN_ID}}")

# --- Sidebar threshold & batch ---
st.sidebar.header("App Settings")
thr = st.sidebar.slider("Decision threshold", 0.0, 1.0, 0.5, 0.01)

st.sidebar.markdown("### Batch Scoring")
uploaded = st.sidebar.file_uploader("Upload CSV with required features", type=["csv"])
if uploaded is not None:
    try:
        df_up = pd.read_csv(uploaded)
        df_val = validate_and_cast(df_up)
        proba = PIPE.predict_proba(df_val)[:, 1]
        pred = (proba >= thr).astype(int)
        out = df_up.copy()
        out["proba"] = proba
        out["pred"] = pred
        st.sidebar.success("Batch scored! Download below.")
        st.sidebar.download_button("Download scored CSV", out.to_csv(index=False), file_name="scored.csv")
    except Exception as e:
        st.sidebar.error(f"Batch scoring failed: {{e}}")

# --- Single prediction form ---
st.subheader("Single Prediction")

defaults = {{
    "age": 57, "sex": 1, "cp": 2, "trestbps": 130, "chol": 250,
    "fbs": 0, "restecg": 1, "thalach": 150, "exang": 0,
    "oldpeak": 1.0, "slope": 1, "ca": 0.0, "thal": 2
}}

cols = st.columns(2)
inputs = {{}}
for i, feat in enumerate(FEATURES_ORDER):
    with cols[i % 2]:
        if feat in ["sex","cp","fbs","restecg","exang","slope","thal"]:
            inputs[feat] = st.number_input(feat, value=int(defaults.get(feat, 0)))
        else:
            inputs[feat] = st.number_input(feat, value=float(defaults.get(feat, 0.0)))

if st.button("Predict"):
    try:
        df = pd.DataFrame([inputs])
        df = validate_and_cast(df)
        p = PIPE.predict_proba(df)[:, 1][0]
        y = int(p >= thr)
        st.metric("Predicted probability", f"{{p:.3f}}")
        st.metric("Predicted label", "Disease" if y==1 else "No Disease")
        st.caption("Tip: move the threshold slider to balance sensitivity vs specificity.")
    except Exception as e:
        st.error(f"Prediction failed: {{e}}")
"""

(APP_DIR / "streamlit_app.py").write_text(streamlit_code)
print("✅ Wrote app/streamlit_app.py")

✅ Wrote app/streamlit_app.py


## 3) Quick sanity test (load the pipeline; predict on 1 row)

In [8]:
pipe = joblib.load(artifact_path)
# Basic 1-row sanity (using simple defaults)
sample = pd.DataFrame([{
    "age":57,"sex":1,"cp":2,"trestbps":130,"chol":250,"fbs":0,"restecg":1,
    "thalach":150,"exang":0,"oldpeak":1.0,"slope":1,"ca":0.0,"thal":2
}])[FEATURES_ORDER]
print("Proba:", pipe.predict_proba(sample)[:,1][0])


Proba: 0.3121578216070636


In [10]:
!streamlit run app/streamlit_app.py --server.address=0.0.0.0 --server.port=8501





  You can now view your Streamlit app in your browser.

  URL: http://0.0.0.0:8501

  Stopping...
  Stopping...


## 5) Run Streamlit in Colab

In [14]:
!pip -q install pyngrok psutil
from pyngrok import ngrok
import os, psutil, signal, pathlib, time

In [23]:
from pyngrok import ngrok
import time, subprocess, os, signal, psutil

# Kill previous streamlit if any
for p in psutil.process_iter(attrs=["pid","name","cmdline"]):
    try:
        if p.info["cmdline"] and "streamlit" in " ".join(p.info["cmdline"]):
            os.kill(p.info["pid"], signal.SIGKILL)
    except Exception:
        pass

# Optional: set your ngrok token from secrets (recommended)
token = os.environ.get("NGROK_AUTH_TOKEN", "").strip()

ngrok.set_auth_token(token)  # writes a fresh config with your token
print("✅ ngrok auth set.")

# Start Streamlit (non-blocking)
proc = subprocess.Popen(
    ["streamlit", "run", "app/streamlit_app.py", "--server.address=0.0.0.0", "--server.port=8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

time.sleep(3)  # give it a moment to boot
tunnel = ngrok.connect(8501, "http", bind_tls=True)
public_url = tunnel.public_url
print("🌍 Streamlit URL:", public_url)


✅ ngrok auth set.


ERROR:pyngrok.process.ngrok:t=2025-10-26T00:26:37+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-10-26T00:26:37+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

## 6) Batch demo

In [24]:
# Create a tiny demo CSV with two rows and score them through the PIPE (outside of Streamlit)
demo = pd.DataFrame([
    {"age":63,"sex":1,"cp":3,"trestbps":145,"chol":233,"fbs":1,"restecg":0,"thalach":150,"exang":0,"oldpeak":2.3,"slope":0,"ca":0.0,"thal":1},
    {"age":58,"sex":0,"cp":2,"trestbps":120,"chol":245,"fbs":0,"restecg":1,"thalach":160,"exang":0,"oldpeak":1.0,"slope":2,"ca":0.0,"thal":2},
])[FEATURES_ORDER]
proba = pipe.predict_proba(demo)[:,1]
pred  = (proba >= 0.5).astype(int)
demo_out = demo.copy(); demo_out["proba"] = proba; demo_out["pred"] = pred
demo_out


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,proba,pred
0,63,1,3,145,233,1,0,150,0,2.3,0,0.0,1,0.398692,0
1,58,0,2,120,245,0,1,160,0,1.0,2,0.0,2,0.244744,0


### ✅ Summary
- Created `app/streamlit_app.py` with a clean UI for single & batch predictions
- Verified the best model loads and returns probabilities
- Ran the app locally or in Colab via `pyngrok`

### 🔜 Next
- Add input validation hints/tooltips
- Package for deployment (e.g., Streamlit Community Cloud / Docker)
- Connect to the FastAPI service (Notebook 09) for a unified demo stack
